# M03C · OOP Practice Problems & Mini-Project: Bio Sequence Toolkit
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2024-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M03_OOP/solutions/M03C_OOP_Practice_and_Mini_Project_Solutions.ipynb)

⏱ ~120 min · Difficulty: ★★★ · Prerequisites: M03A (Classes, Methods, Dunders), M03B (Four Pillars, Access Modifiers, ABCs)

---


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Synthesize** all four pillars of OOP (encapsulation, access modifiers, inheritance, polymorphism, abstraction) into an end-to-end bioinformatics software library.
2. **Implement custom domain exception hierarchies** (`BioInformaticsError`, `InvalidAlphabetError`, `CodonLengthError`) for structured defensive programming.
3. **Build custom container classes** that implement Python protocols (`__len__`, `__iter__`, `__getitem__`) to coordinate genomic collections.
4. **Architect the "Bio Sequence Toolkit"** featuring `BiologicalSequence(ABC)`, `DNA`, `RNA`, `Protein`, and `SequenceCollection`.
5. **Simulate enzyme kinetics and splice-site junction validation** using object-oriented abstractions.
6. **Execute a complete genomic pipeline** that loads multi-FASTA files, filters sequences, translates genes, and exports reports.
7. **Validate** complex object-oriented software architectures using a comprehensive assertion test suite (~15 asserts).

---


## 2. Prerequisites

Before starting this notebook, ensure you understand:
- Abstract Base Classes (`abc.ABC`, `@abstractmethod`).
- Access modifiers: public, protected (`_name`), and private (`__name` name mangling).
- Properties: `@property` getters, setters, and read-only attributes.
- Dunder methods (`__str__`, `__repr__`, `__len__`, `__eq__`, `__iter__`, `__getitem__`).
- Completed **M03A (Classes and Objects)** and **M03B (Four Pillars & Access Modifiers)**.

---


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path
from abc import ABC, abstractmethod

# Detect whether we are running in Google Colab or locally
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root
    ROOT = Path.cwd().resolve()
    while not (ROOT / "data").exists() and ROOT.parent != ROOT:
        ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M03C"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python interpreter : {sys.version.split()[0]}")
print(f"Repository ROOT    : {ROOT.resolve()}")
print(f"DATA_DIR exists    : {DATA_DIR.exists()} ({DATA_DIR})")
print(f"OUT_DIR ready      : {OUT_DIR.exists()} ({OUT_DIR})")


---
## 4. Part 1: Advanced OOP Practice Problems

Master the foundational pillars through these 12 curated bioinformatics design challenges:
- **Q1 (★)**: Custom Biological Exception Hierarchy
- **Q2 (★)**: Encapsulated `ProteinRecord`
- **Q3 (★)**: Cellular Taxonomy Inheritance
- **Q4 (★)**: Species Classification & `__str__` Formatting
- **Q5 (★★)**: Composition Pattern (`Chromosome` containing `Gene` objects)
- **Q6 (★★)**: Operator Overloading & Hashing for `Kmer`
- **Q7 (★★)**: Multiple Inheritance with `FASTAExportMixin`
- **Q8 (★★)**: Singleton Pattern for `CodonTableSingleton`
- **Q9 (★★)**: Method Overloading Simulation for `SequenceFilter`
- **Q10 (★★)**: Dynamic Mutation Simulator with Transactional Rollback
- **Q11 (★★★)**: Michaelis-Menten Enzyme Kinetics Simulator
- **Q12 (★★★)**: Spliced Genomic Locus with Exon-Intron GT-AG Boundary Validation

---


### Q1 (★) — Custom Biological Exceptions

**Problem Statement:**  
Create domain-specific custom exception classes for bioinformatics pipelines:
1. Base class `BioInformaticsError(Exception)`: Root exception for all biological errors.
2. Subclass `InvalidAlphabetError(BioInformaticsError)`: Raised when a sequence contains characters outside its defined alphabet.
3. Subclass `CodonLengthError(BioInformaticsError)`: Raised when an open reading frame or codon slice is not a multiple of 3.

Write a function `validate_coding_dna(dna_seq: str) -> None` that:
- Raises `InvalidAlphabetError` if any character is outside `{'A', 'C', 'G', 'T', 'N'}`.
- Raises `CodonLengthError` if `len(dna_seq) % 3 != 0`.

- **Input:** DNA sequence string.
- **Output:** None (or raises custom exception).


In [ ]:
class BioInformaticsError(Exception):
    """Base exception for all bioinformatics errors."""
    pass

class InvalidAlphabetError(BioInformaticsError):
    """Raised when invalid characters are detected in a biological sequence."""
    pass

class CodonLengthError(BioInformaticsError):
    """Raised when a coding sequence length is not a multiple of 3."""
    pass

def validate_coding_dna(dna_seq: str) -> None:
    clean = dna_seq.strip().upper()
    valid_bases = {"A", "C", "G", "T", "N"}
    invalid = set(clean) - valid_bases
    if invalid:
        raise InvalidAlphabetError(f"Sequence contains invalid bases: {invalid}")
    if len(clean) % 3 != 0:
        raise CodonLengthError(f"Sequence length ({len(clean)}) is not a multiple of 3")


**Explanation:**
- Custom exceptions allow caller code to catch specific biological failures (e.g., `except InvalidAlphabetError:`) without catching unrelated system errors.
- Inheriting from a shared base `BioInformaticsError` allows callers to catch all library-specific errors with a single `except BioInformaticsError:` block.


In [ ]:
# ✅ Test Q1
assert issubclass(InvalidAlphabetError, BioInformaticsError)
assert issubclass(CodonLengthError, BioInformaticsError)

validate_coding_dna("ATGAAATTT")  # Valid: should not raise

try:
    validate_coding_dna("ATGAAATT")  # Length 8 (not multiple of 3)
    assert False, "Should raise CodonLengthError"
except CodonLengthError:
    pass

try:
    validate_coding_dna("ATGAAAZZZ")  # Contains 'Z'
    assert False, "Should raise InvalidAlphabetError"
except InvalidAlphabetError:
    pass

print("Q1 passed ✔")


### Q2 (★) — Encapsulated `ProteinRecord`

**Problem Statement:**  
Create a class `ProteinRecord`:
1. `__init__(self, accession: str, sequence: str)`:
   - `accession`: Public string.
   - `__sequence`: **Private** string (name-mangled).
   - Validates that `sequence` contains only standard 20 amino acids. If invalid, raises `ValueError`.
2. Property `sequence`:
   - Getter returning `self.__sequence`.
   - Setter that validates the new sequence before assigning.
3. Method `length(self) -> int`:
   - Returns length of sequence.

- **Input:** Accession and amino acid string.
- **Output:** Encapsulated `ProteinRecord`.


In [ ]:
class ProteinRecord:
    AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")

    def __init__(self, accession: str, sequence: str):
        self.accession = accession
        self.sequence = sequence  # invokes property setter

    @property
    def sequence(self) -> str:
        return self.__sequence

    @sequence.setter
    def sequence(self, new_seq: str):
        clean = new_seq.strip().upper()
        invalid = set(clean) - self.AMINO_ACIDS
        if invalid:
            raise ValueError(f"Invalid amino acid(s): {invalid}")
        self.__sequence = clean

    def length(self) -> int:
        return len(self.__sequence)


**Explanation:**
- `__sequence` is private and name-mangled to `_ProteinRecord__sequence`.
- The setter enforces the 20 amino acid invariant on both construction and subsequent modification.


In [ ]:
# ✅ Test Q2
p = ProteinRecord("P04637", "MEEPQSDPSV")
assert p.accession == "P04637"
assert p.sequence == "MEEPQSDPSV"
assert p.length() == 10
assert "_ProteinRecord__sequence" in p.__dict__

p.sequence = "AAAA"
assert p.sequence == "AAAA"

try:
    p.sequence = "AA12"
    assert False
except ValueError:
    pass

print("Q2 passed ✔")


### Q3 (★) — Cellular Taxonomy Inheritance Hierarchy

**Problem Statement:**  
Create an inheritance hierarchy for biological cells:
1. Base class `Cell`:
   - `__init__(self, cell_type: str, has_nucleus: bool)`: stores `cell_type` and `has_nucleus`.
   - Method `summary(self) -> str`: returns `"{cell_type} | Nucleus: {has_nucleus}"`.
2. Subclass `ProkaryoticCell(Cell)`:
   - `__init__(self, species_name: str)`: calls `super().__init__(cell_type="Prokaryote", has_nucleus=False)`, stores `species_name`.
   - Method `reproduce(self) -> str`: returns `"Binary fission"`.
3. Subclass `EukaryoticCell(Cell)`:
   - `__init__(self, organism_name: str, organelles: list[str])`: calls `super().__init__(cell_type="Eukaryote", has_nucleus=True)`, stores `organism_name` and list of `organelles`.
   - Method `reproduce(self) -> str`: returns `"Mitosis"`.

- **Input:** Cell attributes.
- **Output:** Hierarchical cell instances.


In [ ]:
class Cell:
    def __init__(self, cell_type: str, has_nucleus: bool):
        self.cell_type = cell_type
        self.has_nucleus = has_nucleus

    def summary(self) -> str:
        return f"{self.cell_type} | Nucleus: {self.has_nucleus}"

class ProkaryoticCell(Cell):
    def __init__(self, species_name: str):
        super().__init__(cell_type="Prokaryote", has_nucleus=False)
        self.species_name = species_name

    def reproduce(self) -> str:
        return "Binary fission"

class EukaryoticCell(Cell):
    def __init__(self, organism_name: str, organelles: list[str]):
        super().__init__(cell_type="Eukaryote", has_nucleus=True)
        self.organism_name = organism_name
        self.organelles = list(organelles)

    def reproduce(self) -> str:
        return "Mitosis"


**Explanation:**
- Base class `Cell` encapsulates fundamental cellular properties (`cell_type`, `has_nucleus`).
- Subclasses specialize reproductive modes and organelle structures using `super()`.


In [ ]:
# ✅ Test Q3
prok = ProkaryoticCell("E. coli")
euk = EukaryoticCell("Human Fibroblast", ["mitochondria", "ER", "golgi"])

assert prok.has_nucleus is False
assert prok.reproduce() == "Binary fission"
assert euk.has_nucleus is True
assert euk.reproduce() == "Mitosis"
assert "mitochondria" in euk.organelles
print("Q3 passed ✔")


### Q4 (★) — Species Classification & `__str__` Formatting

**Problem Statement:**  
Create a class `Species`:
1. `__init__(self, genus: str, specific_epithet: str, common_name: str, ncbi_taxid: int)`:
   - Stores capitalized `genus`, lowercase `specific_epithet`, `common_name`, and integer `ncbi_taxid`.
2. Property `binomial_name -> str`:
   - Returns scientific binomial: `"{genus} {specific_epithet}"`.
3. Method `__str__(self) -> str`:
   - Returns `"{binomial_name} ({common_name}) [TaxID: {ncbi_taxid}]"`.
4. Method `__eq__(self, other: object) -> bool`:
   - Returns `True` if `ncbi_taxid` matches.

- **Input:** Taxonomic parameters.
- **Output:** Structured species object.


In [ ]:
class Species:
    def __init__(self, genus: str, specific_epithet: str, common_name: str, ncbi_taxid: int):
        self.genus = genus.strip().capitalize()
        self.specific_epithet = specific_epithet.strip().lower()
        self.common_name = common_name.strip()
        self.ncbi_taxid = int(ncbi_taxid)

    @property
    def binomial_name(self) -> str:
        return f"{self.genus} {self.specific_epithet}"

    def __str__(self) -> str:
        return f"{self.binomial_name} ({self.common_name}) [TaxID: {self.ncbi_taxid}]"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Species):
            return False
        return self.ncbi_taxid == other.ncbi_taxid


**Explanation:**
- Capitalizing genus and lowercasing epithet enforces binomial nomenclature rules.
- TaxID provides an authoritative, unique identifier for equivalence comparison.


In [ ]:
# ✅ Test Q4
h_sapiens = Species("homo", "SAPIENS", "Human", 9606)
assert h_sapiens.binomial_name == "Homo sapiens"
assert str(h_sapiens) == "Homo sapiens (Human) [TaxID: 9606]"
assert h_sapiens == Species("Homo", "sapiens", "Modern Human", 9606)
print("Q4 passed ✔")


### Q5 (★★) — Composition: `Chromosome` Containing `Gene` Objects

**Problem Statement:**  
Implement the **Composition Pattern** ("has-a"):
1. Class `Gene`:
   - `__init__(self, gene_id: str, start: int, end: int)`: stores attributes.
   - Method `length(self) -> int`: returns `end - start + 1`.
2. Class `Chromosome`:
   - `__init__(self, chrom_name: str)`: stores `chrom_name` and private `_genes = []`.
   - Method `add_gene(self, gene: Gene) -> None`: appends `gene` to `_genes`.
   - Method `find_genes_in_range(self, start: int, end: int) -> list[Gene]`:
     - Returns list of genes whose coordinates overlap with `[start, end]`:
       Overlap condition: `max(gene.start, start) <= min(gene.end, end)`.
   - Method `total_coding_bp(self) -> int`: returns sum of lengths of all genes on chromosome.
   - `__len__(self) -> int`: returns count of genes.

- **Input:** Multiple `Gene` instances added to `Chromosome`.
- **Output:** Coordinated chromosome container querying genes.


In [ ]:
class Gene:
    def __init__(self, gene_id: str, start: int, end: int):
        self.gene_id = gene_id
        self.start = start
        self.end = end

    def length(self) -> int:
        return self.end - self.start + 1

class Chromosome:
    def __init__(self, chrom_name: str):
        self.chrom_name = chrom_name
        self._genes = []

    def add_gene(self, gene: Gene) -> None:
        if not isinstance(gene, Gene):
            raise TypeError("Must be a Gene instance")
        self._genes.append(gene)

    def find_genes_in_range(self, start: int, end: int) -> list[Gene]:
        overlapping = []
        for g in self._genes:
            if max(g.start, start) <= min(g.end, end):
                overlapping.append(g)
        return overlapping

    def total_coding_bp(self) -> int:
        return sum(g.length() for g in self._genes)

    def __len__(self) -> int:
        return len(self._genes)


**Explanation:**
- Composition models the biological reality: a Chromosome *has* many Genes, rather than *is a* Gene.
- `max(g.start, start) <= min(g.end, end)` is the universal closed-interval 1D overlap formula.


In [ ]:
# ✅ Test Q5
chr17 = Chromosome("chr17")
g1 = Gene("TP53", 100, 200)   # length 101
g2 = Gene("WRAP53", 250, 400) # length 151
g3 = Gene("BRCA1", 1000, 1500)# length 501

chr17.add_gene(g1)
chr17.add_gene(g2)
chr17.add_gene(g3)

assert len(chr17) == 3
assert chr17.total_coding_bp() == 101 + 151 + 501
in_range = chr17.find_genes_in_range(150, 300)
assert [g.gene_id for g in in_range] == ["TP53", "WRAP53"]
print("Q5 passed ✔")


### Q6 (★★) — Operator Overloading & Hashing for `Kmer`

**Problem Statement:**  
In genomics assembly (de Bruijn graphs), k-mers must be sorted, compared, and stored as dictionary keys or set elements.

Create a class `Kmer`:
1. `__init__(self, sequence: str)`: stores uppercase `sequence` and `k = len(sequence)`.
2. `__eq__(self, other: object) -> bool`: returns `True` if sequences match.
3. `__hash__(self) -> int`: enables storing `Kmer` instances in `set` and `dict` keys (`hash(self.sequence)`).
4. `__lt__(self, other: "Kmer") -> bool`: enables sorting `Kmer` objects alphabetically (`self.sequence < other.sequence`).
5. `__repr__(self) -> str`: returns `f"Kmer('{self.sequence}')"`.

- **Input:** K-mer sequence strings.
- **Output:** Hashable, sortable `Kmer` instances.


In [ ]:
class Kmer:
    def __init__(self, sequence: str):
        self.sequence = sequence.strip().upper()
        self.k = len(self.sequence)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Kmer):
            return False
        return self.sequence == other.sequence

    def __hash__(self) -> int:
        return hash(self.sequence)

    def __lt__(self, other: "Kmer") -> bool:
        if not isinstance(other, Kmer):
            raise TypeError("Cannot compare Kmer with non-Kmer")
        return self.sequence < other.sequence

    def __repr__(self) -> str:
        return f"Kmer('{self.sequence}')"


**Explanation:**
- In Python, if `__eq__` is overridden without defining `__hash__`, Python sets `__hash__ = None`, making the object unhashable! Implementing `__hash__` restores set/dict capabilities.
- `__lt__` enables built-in `sorted([k1, k2, ...])`.


In [ ]:
# ✅ Test Q6
k1 = Kmer("ATG")
k2 = Kmer("atg")
k3 = Kmer("AAA")

assert k1 == k2
assert len({k1, k2}) == 1  # Deduplicated in set via __hash__ and __eq__
sorted_kmers = sorted([k1, k3])
assert sorted_kmers == [k3, k1]  # 'AAA' < 'ATG'
print("Q6 passed ✔")


### Q7 (★★) — Multiple Inheritance with `FASTAExportMixin`

**Problem Statement:**  
Create a mixin class `FASTAExportMixin` that can be added to any sequence model having attributes `seq_id` and `sequence`:
1. `FASTAExportMixin`:
   - Method `to_fasta_string(self, wrap: int = 60) -> str`:
     - Returns standard FASTA format string with `>{self.seq_id}\n` followed by sequence wrapped at `wrap` width.
2. `SimpleDNA(FASTAExportMixin)`:
   - `__init__(self, seq_id: str, sequence: str)`: stores `seq_id` and `sequence`.
   - Inherits `to_fasta_string` from mixin.

- **Input:** Sequence and ID.
- **Output:** Exportable FASTA formatted string via mixin.


In [ ]:
class FASTAExportMixin:
    def to_fasta_string(self, wrap: int = 60) -> str:
        lines = [f">{getattr(self, 'seq_id', 'unknown')}\n"]
        seq = getattr(self, "sequence", "")
        for i in range(0, len(seq), wrap):
            lines.append(seq[i : i + wrap] + "\n")
        return "".join(lines)

class SimpleDNA(FASTAExportMixin):
    def __init__(self, seq_id: str, sequence: str):
        self.seq_id = seq_id
        self.sequence = sequence.upper()


**Explanation:**
- Mixins provide standalone, reusable behaviors that can be mixed into unrelated class hierarchies without coupling.
- Using `getattr()` inside the mixin defensively handles missing attributes.


In [ ]:
# ✅ Test Q7
dna = SimpleDNA("gene_x", "A" * 70)
fasta_out = dna.to_fasta_string(wrap=60)

assert fasta_out.startswith(">gene_x\n")
lines = fasta_out.strip().splitlines()
assert len(lines) == 3  # header + 60bp + 10bp
assert len(lines[1]) == 60
assert len(lines[2]) == 10
print("Q7 passed ✔")


### Q8 (★★) — The Singleton Design Pattern for `CodonTableSingleton`

**Problem Statement:**  
In high-throughput bioinformatics, lookup tables (like the standard genetic code) should only exist once in memory.

Implement the **Singleton Pattern** by overriding `__new__`:
1. Class `CodonTableSingleton`:
   - Private class attribute `_instance = None`.
   - In `__new__(cls, *args, **kwargs)`:
     - If `cls._instance is None`: creates new instance via `super().__new__(cls)`.
     - Returns `cls._instance`.
   - `__init__(self)`:
     - Initializes `self.table` with a codon mapping if not already initialized.
   - Method `lookup(self, codon: str) -> str`: returns amino acid translation.

- **Input:** Multiple calls to `CodonTableSingleton()`.
- **Output:** Exactly ONE instance shared across all calls (`inst1 is inst2`).


In [ ]:
class CodonTableSingleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def __init__(self):
        if not self._initialized:
            self.table = {
                "AUG": "M", "UUU": "F", "UUC": "F", "UAA": "*", "UAG": "*", "UGA": "*"
            }
            self._initialized = True

    def lookup(self, codon: str) -> str:
        return self.table.get(codon.upper(), "X")


**Explanation:**
- `__new__` controls object allocation before `__init__` is invoked.
- Storing the allocated object in `cls._instance` guarantees that subsequent constructor calls return the exact same memory instance.


In [ ]:
# ✅ Test Q8
t1 = CodonTableSingleton()
t2 = CodonTableSingleton()

assert t1 is t2, "Singleton instances must be identical in memory!"
assert t1.lookup("AUG") == "M"
print("Q8 passed ✔")


### Q9 (★★) — Method Overloading Simulation for `SequenceFilter`

Python does not natively support method overloading with multiple signatures. Instead, Python developers simulate overloading using default arguments and type dispatching.

**Problem Statement:**  
Create a class `SequenceFilter`:
1. `__init__(self, sequences: list[str])`: stores list of uppercase sequence strings.
2. Method `filter_by(self, criterion: int | float | str) -> list[str]`:
   - If `criterion` is `int`: filters sequences with `len(seq) >= criterion`.
   - If `criterion` is `float`: filters sequences with GC fraction $\ge criterion$ (where $0.0 \le 	ext{criterion} \le 1.0$).
   - If `criterion` is `str`: filters sequences that contain the substring `criterion.upper()`.
   - If any other type is passed, raises `TypeError`.

- **Input:** Integer, float, or string filter criterion.
- **Output:** Filtered list of sequence strings.


In [ ]:
class SequenceFilter:
    def __init__(self, sequences: list[str]):
        self.sequences = [s.strip().upper() for s in sequences if s.strip()]

    def filter_by(self, criterion: int | float | str) -> list[str]:
        if isinstance(criterion, int):
            return [s for s in self.sequences if len(s) >= criterion]
        elif isinstance(criterion, float):
            return [
                s for s in self.sequences
                if (s.count("G") + s.count("C")) / len(s) >= criterion
            ]
        elif isinstance(criterion, str):
            motif = criterion.upper()
            return [s for s in self.sequences if motif in s]
        else:
            raise TypeError(f"Unsupported filter criterion type: {type(criterion)}")


**Explanation:**
- Checking `isinstance()` inside `filter_by()` simulates function overloading across integer lengths, float GC thresholds, and string motif queries.


In [ ]:
# ✅ Test Q9
sf = SequenceFilter(["ATGC", "GGCCGGCC", "TTTT", "AAAGGG"])

assert sf.filter_by(6) == ["GGCCGGCC", "AAAGGG"]  # length >= 6
assert sf.filter_by(0.8) == ["GGCCGGCC"]         # GC >= 80%
assert sf.filter_by("ATG") == ["ATGC"]           # contains "ATG"

try:
    sf.filter_by([1, 2, 3])
    assert False
except TypeError:
    pass

print("Q9 passed ✔")


### Q10 (★★) — Dynamic Sequence Mutation Simulator with Rollback

**Problem Statement:**  
Create a class `EvolvingSequence` that simulates point mutations and supports an undo/rollback history:
1. `__init__(self, seq_id: str, initial_sequence: str)`:
   - Stores `seq_id`.
   - `_current_seq`: list of character characters.
   - `_history`: list of previous sequence strings.
2. Property `sequence -> str`:
   - Returns current sequence as string: `"".join(self._current_seq)`.
3. Method `mutate(self, position: int, new_base: str) -> None`:
   - `position`: 1-based coordinate.
   - Saves current sequence to `_history`.
   - Replaces base at `position - 1` with `new_base.upper()`.
4. Method `rollback(self) -> bool`:
   - If `_history` is not empty, pops the last sequence, restores it to `_current_seq`, and returns `True`.
   - If `_history` is empty, returns `False`.
5. Method `divergence(self) -> int`:
   - Returns Hamming distance (number of mismatched bases) between current sequence and the original ancestor sequence (`_history[0]` if history exists, else 0).

- **Input:** Initial sequence, mutation coordinates, rollback requests.
- **Output:** Managed mutating sequence with full audit trail.


In [ ]:
class EvolvingSequence:
    def __init__(self, seq_id: str, initial_sequence: str):
        self.seq_id = seq_id
        clean = initial_sequence.strip().upper()
        self._current_seq = list(clean)
        self._ancestor = clean
        self._history = []

    @property
    def sequence(self) -> str:
        return "".join(self._current_seq)

    def mutate(self, position: int, new_base: str) -> None:
        idx = position - 1  # convert to 0-based
        if not (0 <= idx < len(self._current_seq)):
            raise IndexError(f"Position {position} out of bounds for length {len(self._current_seq)}")
        
        # Save snapshot to history
        self._history.append(self.sequence)
        self._current_seq[idx] = new_base.upper()

    def rollback(self) -> bool:
        if not self._history:
            return False
        prev_seq = self._history.pop()
        self._current_seq = list(prev_seq)
        return True

    def divergence(self) -> int:
        return sum(1 for a, b in zip(self._ancestor, self.sequence) if a != b)


**Explanation:**
- Storing sequence state as a `list` of characters allows $O(1)$ point mutation assignments.
- Maintaining an internal `_history` stack enables robust transactional rollback operations.


In [ ]:
# ✅ Test Q10
evo = EvolvingSequence("sim_1", "ATGCGT")
assert evo.sequence == "ATGCGT"
assert evo.divergence() == 0

# Apply 2 point mutations
evo.mutate(1, "G")  # GTGCGT
evo.mutate(6, "A")  # GTGCGA

assert evo.sequence == "GTGCGA"
assert evo.divergence() == 2

# Rollback one step
assert evo.rollback() is True
assert evo.sequence == "GTGCGT"
assert evo.divergence() == 1

# Rollback to original
assert evo.rollback() is True
assert evo.sequence == "ATGCGT"
assert evo.divergence() == 0

# Rollback when empty
assert evo.rollback() is False
print("Q10 passed ✔")


### Q11 (★★★) — Michaelis-Menten Enzyme Kinetics Simulator

In biochemistry, the Michaelis-Menten model describes the rate of enzymatic reactions:
$$v_0 = \frac{V_{\max} [S]}{K_m + [S]}$$
where $[S]$ is substrate concentration, $V_{\max}$ is maximum reaction velocity, and $K_m$ is the Michaelis constant (substrate concentration at $\frac{1}{2} V_{\max}$).

**Problem Statement:**  
Create a class `MichaelisMentenEnzyme`:
1. `__init__(self, name: str, vmax: float, km: float, enzyme_conc: float)`:
   - Validates that `vmax > 0`, `km > 0`, and `enzyme_conc > 0`. If not, raises `ValueError`.
   - Stores `name`, `vmax` ($\mu\text{M}/\text{s}$), `km` ($\mu\text{M}$), `enzyme_conc` ($\mu\text{M}$).
2. Method `reaction_velocity(self, substrate_conc: float) -> float`:
   - Validates `substrate_conc >= 0`.
   - Calculates $v_0 = \frac{V_{\max} [S]}{K_m + [S]}$, rounded to 3 decimals.
3. Property `kcat -> float`:
   - Turnover number: $k_{\text{cat}} = \frac{V_{\max}}{[E]}$, rounded to 2 decimals.
4. Property `catalytic_efficiency -> float`:
   - Specificity constant: $\frac{k_{\text{cat}}}{K_m}$, rounded to 2 decimals.

- **Input:** Enzyme parameters and substrate concentration.
- **Output:** Initial reaction velocities and catalytic constants.


In [ ]:
class MichaelisMentenEnzyme:
    def __init__(self, name: str, vmax: float, km: float, enzyme_conc: float):
        if vmax <= 0 or km <= 0 or enzyme_conc <= 0:
            raise ValueError("vmax, km, and enzyme_conc must all be strictly positive")
        self.name = name
        self.vmax = float(vmax)
        self.km = float(km)
        self.enzyme_conc = float(enzyme_conc)

    def reaction_velocity(self, substrate_conc: float) -> float:
        s = float(substrate_conc)
        if s < 0:
            raise ValueError(f"Substrate concentration cannot be negative: {s}")
        v0 = (self.vmax * s) / (self.km + s)
        return round(v0, 3)

    @property
    def kcat(self) -> float:
        return round(self.vmax / self.enzyme_conc, 2)

    @property
    def catalytic_efficiency(self) -> float:
        return round(self.kcat / self.km, 2)


**Explanation:**
- $k_{\text{cat}}$ measures catalytic events per second per active site.
- $k_{\text{cat}} / K_m$ reflects catalytic efficiency, with the diffusion limit around $10^8 - 10^9 \text{ M}^{-1}\text{s}^{-1}$.


In [ ]:
# ✅ Test Q11
# Beta-galactosidase model: Vmax=100 uM/s, Km=50 uM, [E]=2 uM
beta_gal = MichaelisMentenEnzyme("Beta-Galactosidase", vmax=100.0, km=50.0, enzyme_conc=2.0)

assert beta_gal.reaction_velocity(50.0) == 50.0  # At [S] = Km, v0 = Vmax / 2
assert beta_gal.reaction_velocity(0.0) == 0.0
assert beta_gal.reaction_velocity(1000.0) > 95.0 # Approaching Vmax
assert beta_gal.kcat == 50.0                     # 100 / 2
assert beta_gal.catalytic_efficiency == 1.0       # 50 / 50

try:
    MichaelisMentenEnzyme("Bad", -10, 50, 2)
    assert False
except ValueError:
    pass

print("Q11 passed ✔")


### Q12 (★★★) — Spliced Genomic Locus with Splice-Junction Validation

In eukaryotic genes, pre-mRNA splicing adheres strictly to the canonical **Chambon Rule (GT-AG Rule)**:
- Introns almost universally begin with the donor dinucleotide `GT` (at the 5′ splice site).
- Introns terminate with the acceptor dinucleotide `AG` (at the 3′ splice site).

**Problem Statement:**  
Create a class `GenomicLocus`:
1. `__init__(self, locus_id: str, genomic_seq: str, exon_coords: list[tuple[int, int]])`:
   - `genomic_seq`: Raw genomic DNA sequence string.
   - `exon_coords`: List of `(start, end)` tuples in 0-based half-open coordinates `[start, end)`.
   - Stores sorted exon coordinates.
2. Method `spliced_mrna(self) -> str`:
   - Extracts and joins all exon slices into a continuous mRNA string.
3. Method `validate_splice_junctions(self) -> list[tuple[int, bool, str]]`:
   - For every consecutive pair of exons $(\text{exon}_i, \text{exon}_{i+1})$:
     - The intervening intron is `genomic_seq[exon_i.end : exon_{i+1}.start]`.
     - Checks whether the intron starts with `"GT"` and ends with `"AG"`.
     - Appends `(intron_index_1_based, is_valid_bool, intron_seq)`.
   - Returns list of validation tuples.

- **Input:** Genomic DNA string and exon coordinates.
- **Output:** Extracted mRNA and GT-AG splice-junction audit results.


In [ ]:
class GenomicLocus:
    def __init__(self, locus_id: str, genomic_seq: str, exon_coords: list[tuple[int, int]]):
        self.locus_id = locus_id
        self.genomic_seq = genomic_seq.strip().upper()
        self.exons = sorted(exon_coords, key=lambda x: x[0])

    def spliced_mrna(self) -> str:
        return "".join(self.genomic_seq[start:end] for start, end in self.exons)

    def validate_splice_junctions(self) -> list[tuple[int, bool, str]]:
        audit_results = []
        for i in range(len(self.exons) - 1):
            intron_start = self.exons[i][1]
            intron_end = self.exons[i + 1][0]
            intron_seq = self.genomic_seq[intron_start:intron_end]
            
            is_valid = (
                len(intron_seq) >= 4 and
                intron_seq.startswith("GT") and
                intron_seq.endswith("AG")
            )
            audit_results.append((i + 1, is_valid, intron_seq))
            
        return audit_results


**Explanation:**
- Intron excision connects exons together to form mature mRNA.
- Validating the GT donor and AG acceptor dinucleotides automates quality control for annotation pipelines.


In [ ]:
# ✅ Test Q12
# Synthetic gene: Exon1 (0..6), Intron1 (6..16 with GT..AG), Exon2 (16..22)
dna = "ATGCGA" + "GTCCCCACAG" + "TTAGCT"
locus = GenomicLocus("TEST_GENE", dna, [(16, 22), (0, 6)])

assert locus.spliced_mrna() == "ATGCGATTAGCT"
junctions = locus.validate_splice_junctions()
assert len(junctions) == 1
intron_num, is_valid, seq = junctions[0]
assert intron_num == 1
assert is_valid is True
assert seq == "GTCCCCACAG"
print("Q12 passed ✔")


---
## 5. Part 2: Mini-Project — "Bio Sequence Toolkit"

In this capstone mini-project, you will design and implement a production-grade object-oriented biological sequence analysis toolkit:
```
BiologicalSequence (ABC)
├── DNA
├── RNA
└── Protein

SequenceCollection (Container managing collections of BiologicalSequence)
```

### Architecture Specifications

#### 1. `BiologicalSequence(ABC)`
- Inherits from `abc.ABC`.
- `__init__(self, seq_id: str, seq: str, source_file: str = "")`:
  - `self.__id = str(seq_id)` (**Private attribute**)
  - `self._source_file = str(source_file)` (**Protected attribute**)
  - Sets sequence via `self.seq = seq` (triggers setter).
- Property `seq_id`: Getter returning private `self.__id`.
- Property `seq`:
  - Getter returning `self._seq`.
  - Setter validating that all characters belong to `self.alphabet`. Raises `ValueError` if invalid.
- Abstract Property `@property @abstractmethod def alphabet(self) -> set[str]: pass`
- Abstract Method `@abstractmethod def molecular_weight(self) -> float: pass`
- Dunder methods:
  - `__len__(self)`: returns `len(self.seq)`
  - `__str__(self)`: returns `f">{self.seq_id} [{self.__class__.__name__}] ({len(self.seq)} bp/aa)"`
  - `__repr__(self)`: returns `f"{self.__class__.__name__}(seq_id={repr(self.seq_id)}, len={len(self.seq)})"`
  - `__eq__(self, other)`: returns `True` if types match and sequences are equal.

#### 2. `DNA(BiologicalSequence)`
- `alphabet`: `{"A", "C", "G", "T", "N"}`
- `molecular_weight(self) -> float`:
  - Nucleotide weights: `{"A": 313.21, "C": 289.18, "G": 329.21, "T": 304.20, "N": 308.95}`
  - Sum of residue weights minus water $(N - 1) \times 18.015$ (or `0.0` if empty), rounded to 2 decimals.
- Method `transcribe(self) -> "RNA"`:
  - Replaces `"T"` with `"U"`.
  - Returns new `RNA` instance with same `seq_id` and `source_file`.
- Method `reverse_complement(self) -> "DNA"`:
  - Returns new `DNA` instance with ID `f"{self.seq_id}_rc"`.
- Method `gc_content(self) -> float`:
  - GC percentage rounded to 2 decimals.

#### 3. `RNA(BiologicalSequence)`
- `alphabet`: `{"A", "C", "G", "U", "N"}`
- `molecular_weight(self) -> float`:
  - Nucleotide weights: `{"A": 329.21, "C": 305.18, "G": 345.21, "U": 306.17, "N": 321.44}`
  - Sum minus $(N - 1) \times 18.015$, rounded to 2 decimals.
- Method `translate(self) -> "Protein"`:
  - Translates codons using standard genetic code up to the end or stop codon.
  - Returns new `Protein` instance with ID `f"{self.seq_id}_protein"`.
- Method `reverse_transcribe(self) -> "DNA"`:
  - Replaces `"U"` with `"T"`, returning new `DNA` instance with ID `f"{self.seq_id}_cdna"`.

#### 4. `Protein(BiologicalSequence)`
- `alphabet`: Standard 20 amino acids + `*`.
- `molecular_weight(self) -> float`:
  - Sum of residue weights $+ 18.015\text{ Da}$ for terminal water, rounded to 2 decimals.
- Method `amino_acid_counts(self) -> dict[str, int]`:
  - Returns dictionary of residue frequencies.

#### 5. `SequenceCollection`
- `__init__(self, sequences: list[BiologicalSequence] = None)`
- Method `add_sequence(self, seq_obj: BiologicalSequence) -> None`: validates type and appends.
- Protocols: `__len__`, `__iter__`, `__getitem__`.
- Method `filter_by_length(self, min_len: int = 0, max_len: int = None) -> "SequenceCollection"`:
  - Returns new `SequenceCollection` containing matching records.
- Method `filter_by_type(self, seq_type: type) -> "SequenceCollection"`:
  - Returns new `SequenceCollection` of matching type.
- Method `to_fasta(self, filepath: Path, wrap: int = 60) -> int`:
  - Writes all sequences to FASTA file, returns count.
- Class method `@classmethod from_fasta(cls, filepath: Path, seq_type: type = DNA) -> "SequenceCollection"`:
  - Parses multi-FASTA file (handling wrapped lines) into a new `SequenceCollection`.
- Method `summary_table(self) -> list[dict]`:
  - Returns list of summary dicts with keys: `["id", "type", "length", "mw"]`.

---


In [ ]:
class BiologicalSequence(ABC):
    """Abstract base class representing any biological polymer (DNA, RNA, Protein)."""

    def __init__(self, seq_id: str, seq: str, source_file: str = ""):
        self.__id = str(seq_id)          # Private: name-mangled
        self._source_file = str(source_file)  # Protected
        self.seq = seq                  # Uses setter for validation

    @property
    def seq_id(self) -> str:
        return self.__id

    @property
    def seq(self) -> str:
        return self._seq

    @seq.setter
    def seq(self, value: str):
        clean = value.strip().upper()
        invalid = set(clean) - self.alphabet
        if invalid:
            raise ValueError(f"Invalid character(s) {invalid} for {self.__class__.__name__}")
        self._seq = clean

    @property
    @abstractmethod
    def alphabet(self) -> set[str]:
        """Allowed sequence alphabet."""
        pass

    @abstractmethod
    def molecular_weight(self) -> float:
        """Calculate molecular weight in Daltons."""
        pass

    def __len__(self) -> int:
        return len(self._seq)

    def __str__(self) -> str:
        return f">{self.seq_id} [{self.__class__.__name__}] ({len(self._seq)} bp/aa)"

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(seq_id={repr(self.seq_id)}, len={len(self._seq)})"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, self.__class__):
            return False
        return self.seq == other.seq


class DNA(BiologicalSequence):
    NT_WEIGHTS = {"A": 313.21, "C": 289.18, "G": 329.21, "T": 304.20, "N": 308.95}
    TRANS_TABLE = str.maketrans("ACGTN", "TGCAN")

    @property
    def alphabet(self) -> set[str]:
        return {"A", "C", "G", "T", "N"}

    def molecular_weight(self) -> float:
        if not self.seq:
            return 0.0
        base_sum = sum(self.NT_WEIGHTS[b] for b in self.seq)
        mw = base_sum - (len(self.seq) - 1) * 18.015
        return round(mw, 2)

    def transcribe(self) -> "RNA":
        return RNA(self.seq_id, self.seq.replace("T", "U"), source_file=self._source_file)

    def reverse_complement(self) -> "DNA":
        rc = self.seq.translate(self.TRANS_TABLE)[::-1]
        return DNA(f"{self.seq_id}_rc", rc, source_file=self._source_file)

    def gc_content(self) -> float:
        if not self.seq:
            return 0.0
        gc = self.seq.count("G") + self.seq.count("C")
        return round((gc / len(self.seq)) * 100.0, 2)


class RNA(BiologicalSequence):
    NT_WEIGHTS = {"A": 329.21, "C": 305.18, "G": 345.21, "U": 306.17, "N": 321.44}
    GENETIC_CODE = {
        "AUG": "M", "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
        "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S", "UAU": "Y",
        "UAC": "Y", "UAA": "*", "UAG": "*", "UGU": "C", "UGC": "C",
        "UGA": "*", "UGG": "W", "CUU": "L", "CUC": "L", "CUA": "L",
        "CUG": "L", "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
        "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q", "CGU": "R",
        "CGC": "R", "CGA": "R", "CGG": "R", "AUU": "I", "AUC": "I",
        "AUA": "I", "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
        "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K", "AGU": "S",
        "AGC": "S", "AGA": "R", "AGG": "R", "GUU": "V", "GUC": "V",
        "GUA": "V", "GUG": "V", "GCU": "A", "GCC": "A", "GCA": "A",
        "GCG": "A", "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
        "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G"
    }

    @property
    def alphabet(self) -> set[str]:
        return {"A", "C", "G", "U", "N"}

    def molecular_weight(self) -> float:
        if not self.seq:
            return 0.0
        base_sum = sum(self.NT_WEIGHTS[b] for b in self.seq)
        mw = base_sum - (len(self.seq) - 1) * 18.015
        return round(mw, 2)

    def translate(self) -> "Protein":
        aa_list = []
        for i in range(0, len(self.seq) - 2, 3):
            codon = self.seq[i : i + 3]
            aa = self.GENETIC_CODE.get(codon, "X")
            if aa == "*":
                break
            aa_list.append(aa)
        prot_seq = "".join(aa_list)
        return Protein(f"{self.seq_id}_protein", prot_seq, source_file=self._source_file)

    def reverse_transcribe(self) -> "DNA":
        return DNA(f"{self.seq_id}_cdna", self.seq.replace("U", "T"), source_file=self._source_file)


class Protein(BiologicalSequence):
    AA_WEIGHTS = {
        'A': 71.08, 'C': 103.14, 'D': 115.09, 'E': 129.12, 'F': 147.18,
        'G': 57.05, 'H': 137.14, 'I': 113.16, 'K': 128.17, 'L': 113.16,
        'M': 131.20, 'N': 114.10, 'P': 97.12,  'Q': 128.13, 'R': 156.19,
        'S': 87.08, 'T': 101.11, 'V': 99.13,  'W': 186.21, 'Y': 163.18,
        '*': 0.0, 'X': 110.0
    }

    @property
    def alphabet(self) -> set[str]:
        return set(self.AA_WEIGHTS.keys())

    def molecular_weight(self) -> float:
        if not self.seq:
            return 0.0
        mw = sum(self.AA_WEIGHTS[aa] for aa in self.seq) + 18.015
        return round(mw, 2)

    def amino_acid_counts(self) -> dict[str, int]:
        counts = {}
        for aa in self.seq:
            counts[aa] = counts.get(aa, 0) + 1
        return counts


class SequenceCollection:
    """Container class coordinating collections of BiologicalSequence objects."""

    def __init__(self, sequences: list[BiologicalSequence] = None):
        self._sequences: list[BiologicalSequence] = []
        if sequences is not None:
            for s in sequences:
                self.add_sequence(s)

    def add_sequence(self, seq_obj: BiologicalSequence) -> None:
        if not isinstance(seq_obj, BiologicalSequence):
            raise TypeError("Object must inherit from BiologicalSequence")
        self._sequences.append(seq_obj)

    def __len__(self) -> int:
        return len(self._sequences)

    def __iter__(self):
        return iter(self._sequences)

    def __getitem__(self, index):
        return self._sequences[index]

    def filter_by_length(self, min_len: int = 0, max_len: int = None) -> "SequenceCollection":
        filtered = []
        for s in self._sequences:
            if len(s) >= min_len:
                if max_len is None or len(s) <= max_len:
                    filtered.append(s)
        return SequenceCollection(filtered)

    def filter_by_type(self, seq_type: type) -> "SequenceCollection":
        return SequenceCollection([s for s in self._sequences if isinstance(s, seq_type)])

    def to_fasta(self, filepath: Path, wrap: int = 60) -> int:
        with open(filepath, "w", encoding="utf-8") as f:
            for s in self._sequences:
                f.write(f">{s.seq_id}\n")
                for i in range(0, len(s.seq), wrap):
                    f.write(s.seq[i : i + wrap] + "\n")
        return len(self._sequences)

    @classmethod
    def from_fasta(cls, filepath: Path, seq_type: type = DNA) -> "SequenceCollection":
        records = []
        current_id = None
        chunks = []

        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                cleaned = line.strip()
                if not cleaned:
                    continue
                if cleaned.startswith(">"):
                    if current_id is not None:
                        seq_str = "".join(chunks)
                        records.append(seq_type(current_id, seq_str, source_file=filepath.name))
                    current_id = cleaned[1:].strip().split()[0]
                    chunks = []
                else:
                    chunks.append(cleaned)

            if current_id is not None:
                seq_str = "".join(chunks)
                records.append(seq_type(current_id, seq_str, source_file=filepath.name))

        return cls(records)

    def summary_table(self) -> list[dict]:
        return [
            {
                "id": s.seq_id,
                "type": s.__class__.__name__,
                "length": len(s),
                "mw": s.molecular_weight()
            }
            for s in self._sequences
        ]


**Architecture Explanation:**
- **Abstraction**: `BiologicalSequence` defines the interface contract (`alphabet`, `molecular_weight()`), forbidding direct instantiation while ensuring all biological sequence types expose consistent behaviors.
- **Encapsulation & Access Modifiers**: Private `__id` prevents accidental ID mutation; protected `_source_file` permits subclass logging; the `@seq.setter` validates incoming characters against the subclass `alphabet`.
- **Inheritance**: `DNA`, `RNA`, and `Protein` inherit shared protocols (`__len__`, `__str__`, `__repr__`, `__eq__`) while specializing their chemical alphabets, molecular weights, and transformation methods.
- **Polymorphism & Duck Typing**: `SequenceCollection` operates uniformly over any sequence type, enabling polymorphic filtering, FASTA exports, and tabular reports.


In [ ]:
# ✅ Comprehensive Test Suite (~15 asserts for Bio Sequence Toolkit)

# 1. Abstraction: BiologicalSequence cannot be instantiated directly
try:
    BiologicalSequence("test", "ATGC")
    assert False, "Assert 1 Failed: Should not instantiate ABC directly"
except TypeError:
    pass

# 2. Validation: Invalid DNA characters raise ValueError
try:
    DNA("bad", "ATGCZ123")
    assert False, "Assert 2 Failed: Should reject invalid alphabet"
except ValueError:
    pass

# 3. Construction & Property Getter: seq_id
d = DNA("sample_test", "ATGCGTACGTTAGC")
assert d.seq_id == "sample_test", f"Assert 3 Failed: ID is {d.seq_id}"

# 4. Access Modifiers: Name mangling of private __id
assert "_BiologicalSequence__id" in d.__dict__, "Assert 4 Failed: Private __id must be mangled"
assert "__id" not in d.__dict__, "Assert 4 Failed: Raw __id must not exist in dict"

# 5. Transcription: DNA -> RNA
r = d.transcribe()
assert isinstance(r, RNA), "Assert 5 Failed: transcribe() must return RNA"
assert "U" in r.seq and "T" not in r.seq, "Assert 5 Failed: RNA must contain U, not T"

# 6. Translation: RNA -> Protein
p = r.translate()
assert isinstance(p, Protein), "Assert 6 Failed: translate() must return Protein"
assert len(p) == len(r) // 3, f"Assert 6 Failed: Protein length mismatch: {len(p)}"

# 7. Reverse Complement
d_rc = d.reverse_complement()
assert isinstance(d_rc, DNA), "Assert 7 Failed: reverse_complement must return DNA"
assert d_rc.seq == "GCTAACGTACGCAT", f"Assert 7 Failed: Expected GCTAACGTACGCAT, got {d_rc.seq}"

# 8. GC Content Calculation
assert d.gc_content() == 50.0, f"Assert 8 Failed: Expected 50.0% GC, got {d.gc_content()}"

# 9. DNA Molecular Weight > 0
assert d.molecular_weight() > 4000.0, f"Assert 9 Failed: MW was {d.molecular_weight()}"

# 10. Protein Molecular Weight > 0
assert p.molecular_weight() > 400.0, f"Assert 10 Failed: Protein MW was {p.molecular_weight()}"

# 11. SequenceCollection Factory: from_fasta loads 15 sequences
fasta_path = DATA_DIR / "sequences" / "multi_sample.fasta"
col = SequenceCollection.from_fasta(fasta_path, seq_type=DNA)
assert len(col) == 15, f"Assert 11 Failed: Expected 15 sequences, got {len(col)}"

# 12. SequenceCollection Filtering by length (in multi_sample.fasta, 7 seqs >= 200 bp)
long_col = col.filter_by_length(min_len=200)
assert len(long_col) == 7, f"Assert 12 Failed: Expected 7 sequences >= 200 bp, got {len(long_col)}"

# 13. Indexing and Iteration protocols
assert col[0].seq_id == "sample_01", f"Assert 13 Failed: col[0] ID is {col[0].seq_id}"
iterated_count = sum(1 for _ in col)
assert iterated_count == 15, f"Assert 13 Failed: Iteration count {iterated_count} != 15"

# 14. Export to FASTA creates valid file
out_test_fasta = OUT_DIR / "exported_toolkit_test.fasta"
exported_n = col.to_fasta(out_test_fasta)
assert exported_n == 15, f"Assert 14 Failed: Expected 15 exported, got {exported_n}"
assert out_test_fasta.exists() and out_test_fasta.stat().st_size > 1000

# 15. Summary Table generation
summary = col.summary_table()
assert len(summary) == 15, f"Assert 15 Failed: Expected 15 summary rows, got {len(summary)}"
assert all("id" in row and "type" in row and "length" in row and "mw" in row for row in summary)

print("All 15 assertions in Bio Sequence Toolkit test suite passed cleanly ✔")


---
## 6. Summary & Cheat Sheet

| Component | Responsibility / Protocol | Design Pattern |
|---|---|---|
| `BiologicalSequence` | Abstract root defining `alphabet`, `molecular_weight()`, and dunders | **Interface / Abstraction** |
| `DNA` | Encapsulates DNA state, `.transcribe()`, `.reverse_complement()` | **Subclass Specialization** |
| `RNA` | Encapsulates RNA state, `.translate()` using standard codon table | **Subclass Specialization** |
| `Protein` | Encapsulates peptide sequence, residue weights, amino acid counts | **Subclass Specialization** |
| `SequenceCollection` | Container managing collections of sequences (`__len__`, `__iter__`, `__getitem__`) | **Composite / Container** |
| `from_fasta` | Alternative constructor parsing raw files into high-level objects | **Factory Method** |
| `summary_table` | Serializes collection state into structured records | **Data Pipeline Export** |

---


## 7. Further Reading & Resources

- [Gamma et al.: Design Patterns — Elements of Reusable Object-Oriented Software (Gang of Four)](https://en.wikipedia.org/wiki/Design_Patterns)
- [Python Documentation: Data Model & Custom Containers](https://docs.python.org/3/reference/datamodel.html#emulating-container-types)
- [Biopython `Bio.SeqRecord` Design Architecture](https://biopython.org/wiki/SeqRecord)
- [Fluent Python by Luciano Ramalho (Chapter on Object-Oriented Idioms)](https://www.oreilly.com/library/view/fluent-python-2nd/9781492056348/)
